# 03. 고속도로 선형(OSM) → 1km 구간, 주행 방위각, 터널 위치

입력: `data/interim/accident_clean.csv`, `data/interim/tunnel_expressway.csv`
출력: `data/processed/chains.pkl`(주행 방향별 고속도로 선), `data/processed/segments.csv`(1km 구간), `data/processed/accident_snapped.csv`

**왜 OSM을 쓰나요?**
눈부심은 "차가 향하는 방향"과 "해의 방향"이 얼마나 겹치는지가 핵심이에요. 사고 데이터에는 좌표만 있고 차가 어느 쪽을 보고 달렸는지는 없어서,
OpenStreetMap의 고속도로 선형(상·하행이 따로 그려져 있음)을 받아 각 지점의 주행 방위각을 구해요.
OSM에는 터널 구간도 표시돼 있어서, 주행 방향 기준으로 **입구까지 남은 거리 / 출구 이후 거리**도 정확히 계산할 수 있어요.

용어
- **chain**: 한 방향으로 끊김 없이 이어진 고속도로 차로 묶음 (예: 경부선 부산방향의 한 구간)
- **s**: chain 시작점부터 주행 방향으로 몇 m 왔는지
- **segment**: chain을 약 1km씩 자른 학습 단위 (방향이 이미 포함됨)

In [ ]:
%pip install osmnx geopandas -q

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString, Point
from pyproj import Geod

ROOT = Path.cwd()
INTERIM, EXTERNAL, PROCESSED = (ROOT / "data" / d for d in ["interim", "external", "processed"])
for d in [EXTERNAL, PROCESSED]:
    d.mkdir(parents=True, exist_ok=True)
GEOD = Geod(ellps="WGS84")
CRS_M = 5179   # 한국 미터 좌표계

## 1. OSM 고속도로 받기 (처음 한 번, 몇 분 걸림)
`highway=motorway`만 받아요(나들목 램프 제외). 받은 결과는 `data/external/osm_motorway_edges.pkl`에 저장돼서 다음부터는 바로 읽어요.

In [ ]:
EDGE_CACHE = EXTERNAL / "osm_motorway_edges.pkl"
if not EDGE_CACHE.exists():
    import osmnx as ox
    ox.settings.use_cache = True
    ox.settings.cache_folder = str(EXTERNAL / "osm_cache")
    G = ox.graph_from_place("South Korea", custom_filter='["highway"="motorway"]',
                            simplify=False, retain_all=True)
    e = ox.graph_to_gdfs(G, nodes=False).reset_index()
    keep = [c for c in ["u", "v", "ref", "name", "tunnel", "maxspeed", "geometry"] if c in e.columns]
    e[keep].to_pickle(EDGE_CACHE)
edges = pd.read_pickle(EDGE_CACHE)
for c in ["ref", "name", "tunnel", "maxspeed"]:
    if c not in edges.columns:
        edges[c] = None
    edges[c] = edges[c].apply(lambda x: x[0] if isinstance(x, list) else x)
edges = gpd.GeoDataFrame(edges, geometry="geometry", crs=4326).to_crs(CRS_M).reset_index(drop=True)
edges["is_tunnel"] = edges["tunnel"].notna() & ~edges["tunnel"].isin(["no", "False", False])
print(f"OSM 고속도로 조각 {len(edges):,}개, 그중 터널 {edges['is_tunnel'].sum():,}개")

## 2. 조각을 주행 방향으로 이어 chain 만들기
OSM에서 고속도로는 진행 방향(일방통행)으로 그려져 있어서, 끝점이 다음 조각의 시작점이 되도록 이어 붙이면 됩니다.
분기·합류 지점에서는 chain이 끊겨요.

In [ ]:
u, v = edges["u"].to_numpy(), edges["v"].to_numpy()
out_deg = pd.Series(u).value_counts()
in_deg = pd.Series(v).value_counts()
next_edge = pd.Series(edges.index, index=u).groupby(level=0).first()
interior = lambda n: in_deg.get(n, 0) == 1 and out_deg.get(n, 0) == 1

visited = np.zeros(len(edges), bool)
def walk(i):
    seq = []
    while not visited[i]:
        seq.append(i); visited[i] = True
        n = v[i]
        if not interior(n):
            break
        i = next_edge[n]
    return seq

chain_edges = [walk(i) for i in edges.index if not visited[i] and not interior(u[i])]
chain_edges += [walk(i) for i in edges.index if not visited[i]]      # 고리 모양 남은 것

rows = []
for cid, seq in enumerate(chain_edges):
    coords, tunnels, s = [], [], 0.0
    for i in seq:
        g = edges.geometry[i]
        pts = list(g.coords)
        coords.extend(pts if not coords else pts[1:])
        if edges.is_tunnel[i]:
            if tunnels and abs(tunnels[-1][1] - s) < 1:
                tunnels[-1][1] = s + g.length         # 이어지는 터널 조각은 합침
            else:
                tunnels.append([s, s + g.length])
        s += g.length
    if len(coords) < 2:
        continue
    sub = edges.loc[seq]
    rows.append({"chain_id": cid, "ref": sub["ref"].mode().iloc[0] if sub["ref"].notna().any() else None,
                 "name": sub["name"].mode().iloc[0] if sub["name"].notna().any() else None,
                 "maxspeed": pd.to_numeric(sub["maxspeed"], errors="coerce").median(),
                 "tunnels": [tuple(t) for t in tunnels if t[1] - t[0] >= 30],   # 30m 미만은 노이즈로 봄
                 "geometry": LineString(coords)})
chains = gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS_M)
chains["length_m"] = chains.length
chains.to_pickle(PROCESSED / "chains.pkl")
print(f"chain {len(chains):,}개, 총 {chains.length_m.sum() / 1000:,.0f} km (양방향 합)")
print(f"터널 {chains.tunnels.str.len().sum():,}개")

## 3. 위치 → 위경도, 주행 방위각 함수
방위각은 북=0°, 동=90°, 남=180°, 서=270° (pvlib 태양 방위각과 같은 기준). 해당 지점 앞뒤 50m를 이어 계산해요.

In [ ]:
_to_wgs = gpd.GeoSeries(crs=CRS_M)
def to_lonlat(points):
    g = gpd.GeoSeries(points, crs=CRS_M).to_crs(4326)
    return g.x.to_numpy(), g.y.to_numpy()

def heading_at(line, s, half=50):
    a = line.interpolate(max(s - half, 0)); b = line.interpolate(min(s + half, line.length))
    (lon1, lon2), (lat1, lat2) = to_lonlat([a, b])
    az, _, _ = GEOD.inv(lon1, lat1, lon2, lat2)
    return az % 360

## 4. 약 1km 구간(segment) 만들기

In [ ]:
seg_rows = []
for c in chains.itertuples():
    n = max(1, int(round(c.length_m / 1000)))
    edges_s = np.linspace(0, c.length_m, n + 1)
    for k in range(n):
        s0, s1 = edges_s[k], edges_s[k + 1]
        seg_rows.append({"segment_id": f"{c.chain_id}_{k}", "chain_id": c.chain_id, "s0": s0, "s1": s1,
                         "ref": c.ref, "name": c.name, "maxspeed": c.maxspeed,
                         "mid": c.geometry.interpolate((s0 + s1) / 2)})
segments = pd.DataFrame(seg_rows)
segments["mid_lon"], segments["mid_lat"] = to_lonlat(segments.pop("mid"))
segments["length_m"] = segments.s1 - segments.s0
segments.to_csv(PROCESSED / "segments.csv", index=False, encoding="utf-8-sig")
print(f"segment {len(segments):,}개, 길이 중앙값 {segments.length_m.median():.0f} m")

## 5. 사고를 도로에 붙이기 (어느 방향 차로였는지 정하기)
사고 좌표에서 100m 안의 chain이 보통 2개(상·하행) 나와요. 어느 쪽인지는 `방향` 열로 정합니다.

- 같은 노선의 다른 사고와 **이정(km)**을 비교하면 "이정이 커지는 쪽"의 실제 방위각을 알 수 있어요.
- `방향` 값(예: ○○방향)마다 "이정 증가 쪽 차로에 더 가까이 찍힌 비율"을 보고 증가/감소 방향을 자동으로 판별해요.
- 판별이 안 되면 가장 가까운 차로를 써요. 아래 표에서 결과를 꼭 확인하세요.

In [ ]:
acc = pd.read_csv(INTERIM / "accident_clean.csv", parse_dates=["사고일시"])
acc_pts = gpd.GeoSeries(gpd.points_from_xy(acc.lon, acc.lat), crs=4326).to_crs(CRS_M)

def ang_diff(a, b):
    d = np.abs((np.asarray(a) - np.asarray(b) + 180) % 360 - 180)
    return d

# (1) 이정이 커지는 방향의 방위각: 같은 노선에서 이정 차이 0.5~30km인 가장 가까운 사고와 비교
inc_bearing = np.full(len(acc), np.nan)
for route, g in acc.groupby("사고노선"):
    g = g.dropna(subset=["이정_km"]).sort_values("이정_km")
    km, lat, lon, idx = g.이정_km.to_numpy(), g.lat.to_numpy(), g.lon.to_numpy(), g.index.to_numpy()
    for j in range(len(g)):
        dkm = np.abs(km - km[j])
        ok = (dkm >= 0.5) & (dkm <= 30)
        if not ok.any():
            continue
        p = np.where(ok)[0][np.argmin(dkm[ok])]
        a, b = (j, p) if km[p] > km[j] else (p, j)
        inc_bearing[idx[j]] = GEOD.inv(lon[a], lat[a], lon[b], lat[b])[0] % 360

# (2) 100m 안의 후보 차로들
cand = []
for i, pt in enumerate(acc_pts):
    for cid in chains.sindex.query(pt, predicate="dwithin", distance=100):
        line = chains.geometry.iloc[cid]
        s = line.project(pt)
        cand.append((i, chains.chain_id.iloc[cid], s, line.distance(pt), heading_at(line, s)))
cand = pd.DataFrame(cand, columns=["acc_idx", "chain_id", "s", "dist_m", "heading"])
cand["inc_bearing"] = inc_bearing[cand.acc_idx]
cand["is_inc"] = ang_diff(cand.heading, cand.inc_bearing) < 90
print("100m 안에 도로가 없는 사고:", len(acc) - cand.acc_idx.nunique())

In [ ]:
# (3) 방향 값별로 '이정 증가 차로'에 더 가까이 찍힌 비율
nearest = cand.sort_values("dist_m").groupby("acc_idx").first()
both = cand.groupby("acc_idx").is_inc.nunique() == 2          # 양쪽 차로가 다 후보인 사고만
nearest = nearest[both.reindex(nearest.index, fill_value=False) & nearest.inc_bearing.notna()]
nearest["방향"] = acc.loc[nearest.index, "방향"]
dir_table = nearest.groupby("방향").is_inc.agg(["mean", "size"]).rename(columns={"mean": "이정증가_비율", "size": "건수"})
dir_table["판정"] = np.select([dir_table.이정증가_비율 >= 0.6, dir_table.이정증가_비율 <= 0.4], ["증가", "감소"], "불명")
dir_table

위 표에서 `판정`이 이상하면 아래 `DIR_OVERRIDE`에 직접 적어주세요. 예: `{"부산방향": "증가", "서울방향": "감소"}`

In [ ]:
DIR_OVERRIDE = {}
dir_map = {**dir_table.판정.to_dict(), **DIR_OVERRIDE}

want = acc.방향.map(dir_map).fillna("불명").to_numpy()[cand.acc_idx]
cand["dir_match"] = (want != "불명") & cand.inc_bearing.notna() & (cand.is_inc == (want == "증가"))
# 방향이 맞는 차로 중 가장 가까운 것, 없으면 그냥 가장 가까운 차로
snap = (cand.sort_values(["dir_match", "dist_m"], ascending=[False, True])
            .drop_duplicates("acc_idx").set_index("acc_idx"))
snap["method"] = np.where(snap.dir_match, "방향", "최근접")
out = acc.join(snap[["chain_id", "s", "dist_m", "heading", "method"]], how="inner")
seg_lookup = segments.set_index("chain_id")[["segment_id", "s0", "s1"]]
out["segment_id"] = [
    seg_lookup.loc[[c]].query("s0 <= @s <= s1").segment_id.iloc[0] for c, s in zip(out.chain_id, out.s)]
out.to_csv(PROCESSED / "accident_snapped.csv", index=False, encoding="utf-8-sig")
print(f"{len(acc)} 중 {len(out)}건 도로에 붙임")
print(out.method.value_counts())